# Детектирование приложений в зашифрованном TCP-трафике
Полный Colab-пайплайн: подготовка данных, стратифицированная кросс-валидация, CatBoost, 1D-CNN и submission. Перед запуском выберите **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

# Измените только эти два пути, если папки на Google Drive называются иначе.
DRIVE_PROJECT = Path('/content/drive/MyDrive/Kursach/traffic_detection')
DRIVE_DATA = Path('/content/drive/MyDrive/Kursach/whos-talking-classify-encrypted-tcp-traffic-spring-2')
LOCAL_PROJECT = Path('/content/traffic_detection')
LOCAL_DATA = Path('/content/competition_data')
PREPARED = Path('/content/prepared')

assert (DRIVE_PROJECT / 'scripts').is_dir(), f'Не найдены скрипты: {DRIVE_PROJECT}'
assert (DRIVE_DATA / 'train.csv').is_file(), f'Не найден train.csv: {DRIVE_DATA}'
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, dirs_exist_ok=True)
LOCAL_DATA.mkdir(exist_ok=True)
for name in ('train.csv', 'test.csv'):
    target = LOCAL_DATA / name
    if not target.exists():
        shutil.copy2(DRIVE_DATA / name, target)
print('Проект и данные скопированы на быстрый локальный диск Colab')

In [ ]:
!pip -q install 'catboost>=1.2,<2' 'scikit-learn>=1.5,<2' 'matplotlib>=3.9,<4' tqdm
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите T4 GPU')

## 1. Конвертация CSV и построение признаков
CSV преобразуется в компактные memory-map массивы. Повторный запуск можно пропустить, если папка `/content/prepared` сохранена в текущей сессии.

In [ ]:
if not (PREPARED/'train_features.npy').exists():
    cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/prepare_data.py'),
           '--input-dir', str(LOCAL_DATA), '--output-dir', str(PREPARED),
           '--with-features', '--batch-size', '100000']
    subprocess.run(cmd, check=True)
else:
    print('Подготовленные массивы уже существуют')

## 2A. Быстрая проверка уже обученных весов
Это не заменяет K-fold с переобучением. Скрипт оценивает CNN, CatBoost и ансамбль на пяти стратифицированных частях исходного независимого holdout. Так можно быстро проверить средний F1 и его разброс для уже сохранённых весов.

In [ ]:
SAVED_RESULTS = DRIVE_PROJECT / 'colab_results'
SAVED_EVAL_DIR = Path('/content/artifacts/saved_weights_evaluation')
saved_cnn = SAVED_RESULTS / 'best.pt'
saved_catboost = SAVED_RESULTS / 'model.cbm'
saved_ensemble = SAVED_RESULTS / 'ensemble.json'
if all(path.is_file() for path in (saved_cnn, saved_catboost, saved_ensemble)):
    cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/evaluate_saved_models.py'),
           '--data-dir', str(PREPARED), '--cnn-checkpoint', str(saved_cnn),
           '--catboost-model', str(saved_catboost),
           '--ensemble-config', str(saved_ensemble),
           '--output-dir', str(SAVED_EVAL_DIR),
           '--folds', '5', '--val-size', '0.05', '--seed', '42']
    subprocess.run(cmd, check=True)
else:
    print('Сохранённые веса не найдены в', SAVED_RESULTS, '— пропускаем быструю проверку')

## 2B. Полноценная стратифицированная кросс-валидация
Каждый выбранный поток ровно один раз используется для проверки. Сохраняются результаты каждого фолда, среднее и стандартное отклонение macro-F1, а также OOF-предсказания. Ограничение `cap-per-class` ускоряет оценку и не меняет настройки последующего итогового обучения. Если времени мало, CNN-CV можно запустить с 3 фолдами и 2 эпохами.

In [ ]:
CAT_CV_DIR = Path('/content/artifacts/catboost_cv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/cross_validate_catboost.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(CAT_CV_DIR),
       '--folds', '5', '--cap-per-class', '2000',
       '--iterations', '500', '--task-type', 'GPU']
subprocess.run(cmd, check=True)
shutil.copytree(CAT_CV_DIR, DRIVE_PROJECT/'colab_results'/'catboost_cv', dirs_exist_ok=True)
print('CatBoost CV сохранена на Drive')

In [ ]:
CNN_CV_DIR = Path('/content/artifacts/cnn_cv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/cross_validate_cnn.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(CNN_CV_DIR),
       '--folds', '5', '--cap-per-class', '5000',
       '--epochs', '4', '--batch-size', '4096', '--workers', '2']
subprocess.run(cmd, check=True)
shutil.copytree(CNN_CV_DIR, DRIVE_PROJECT/'colab_results'/'cnn_cv', dirs_exist_ok=True)
print('CNN CV сохранена на Drive')

## 3. CatBoost baseline
Сбалансированная выборка до 2000 объектов каждого класса и естественная стратифицированная validation-выборка.

In [ ]:
CAT_DIR = Path('/content/artifacts/catboost')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/train_catboost.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(CAT_DIR),
       '--cap-per-class', '2000', '--iterations', '500', '--task-type', 'GPU']
subprocess.run(cmd, check=True)

In [ ]:
CAT_SUBMISSION = Path('/content/submission_catboost.csv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/predict_catboost.py'),
       '--data-dir', str(PREPARED), '--model', str(CAT_DIR/'model.cbm'),
       '--output', str(CAT_SUBMISSION)]
subprocess.run(cmd, check=True)

## 4. 1D-CNN
Сеть использует отдельные embeddings размера пакета, направления и позиции. Для первого smoke-test добавьте `--max-train-rows 50000 --epochs 3`; для итогового обучения оставьте команду ниже.

In [ ]:
CNN_DIR = Path('/content/artifacts/cnn')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/train_cnn.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(CNN_DIR),
       '--epochs', '8', '--batch-size', '4096', '--workers', '2', '--val-size', '0.05']
subprocess.run(cmd, check=True)

In [ ]:
CNN_SUBMISSION = Path('/content/submission_cnn.csv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/predict_cnn.py'),
       '--data-dir', str(PREPARED), '--checkpoint', str(CNN_DIR/'best.pt'),
       '--output', str(CNN_SUBMISSION), '--batch-size', '8192']
subprocess.run(cmd, check=True)

## 5. Ансамбль CatBoost + CNN
Вес моделей автоматически выбирается по macro-F1 на общей validation-выборке. Вероятности test обрабатываются пакетами и не занимают гигабайты на диске.

In [ ]:
ENSEMBLE_CONFIG = Path('/content/artifacts/ensemble.json')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/tune_ensemble.py'),
       '--data-dir', str(PREPARED), '--catboost-dir', str(CAT_DIR),
       '--cnn-dir', str(CNN_DIR), '--output', str(ENSEMBLE_CONFIG)]
subprocess.run(cmd, check=True)
ENSEMBLE_SUBMISSION = Path('/content/submission_ensemble.csv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/predict_ensemble.py'),
       '--data-dir', str(PREPARED), '--catboost-model', str(CAT_DIR/'model.cbm'),
       '--cnn-checkpoint', str(CNN_DIR/'best.pt'),
       '--ensemble-config', str(ENSEMBLE_CONFIG), '--output', str(ENSEMBLE_SUBMISSION)]
subprocess.run(cmd, check=True)

## 6. Exact lookup override
Около 10% тестовых последовательностей полностью присутствуют в train. Для них заменяем ответ модели на известную обучающую метку.

In [ ]:
LOOKUP_SUBMISSION = Path('/content/submission_final.csv')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/apply_exact_lookup.py'),
       '--data-dir', str(PREPARED), '--base-submission', str(ENSEMBLE_SUBMISSION),
       '--output', str(LOOKUP_SUBMISSION)]
subprocess.run(cmd, check=True)

## 7. Статистика и графики для отчёта
Ячейка анализирует весь подготовленный train пакетами, поэтому не переполняет оперативную память. Она строит графики по меткам, длинам потоков, размерам пакетов, пяти фолдам, эпохам CNN и итоговым моделям.

In [ ]:
REPORT_ASSETS_DIR = Path('/content/artifacts/report_assets')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/generate_report_assets.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(REPORT_ASSETS_DIR),
       '--catboost-cv', str(CAT_CV_DIR/'catboost_cv.json'),
       '--cnn-cv', str(CNN_CV_DIR/'cnn_cv.json'),
       '--catboost-metrics', str(CAT_DIR/'metrics.json'),
       '--cnn-history', str(CNN_DIR/'history.json'),
       '--ensemble', str(ENSEMBLE_CONFIG), '--batch-size', '100000']
subprocess.run(cmd, check=True)
print('Графики:', sorted(path.name for path in REPORT_ASSETS_DIR.glob('*.png')))

## 8. Паттерны конкретных приложений
Отдельно строим профили всех 252 классов: длина потока, направления, размеры, смены направления, позиционные паттерны, ближайшие приложения и пять описательных кластеров. Обучение моделей в этой ячейке не выполняется.

In [ ]:
PATTERN_ANALYSIS_DIR = Path('/content/artifacts/application_analysis')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/analyze_application_patterns.py'),
       '--data-dir', str(PREPARED), '--output-dir', str(PATTERN_ANALYSIS_DIR),
       '--batch-size', '250000', '--clusters', '5']
subprocess.run(cmd, check=True)
print('Аналитические таблицы:', sorted(path.name for path in PATTERN_ANALYSIS_DIR.glob('*.csv')))
print('Графики паттернов:', sorted(path.name for path in PATTERN_ANALYSIS_DIR.glob('*.png')))

## 9. ROC, Precision–Recall и анализ ошибок
На стратифицированной части исходного независимого holdout вычисляются вероятностные метрики One-vs-Rest, top-k accuracy, калибровка, per-class F1, confusion matrix и наиболее частые пары ошибок. Модели не переобучаются.

In [ ]:
EVALUATION_DIAGNOSTICS_DIR = Path('/content/artifacts/evaluation_diagnostics')
cmd = [sys.executable, str(LOCAL_PROJECT/'scripts/generate_evaluation_diagnostics.py'),
       '--data-dir', str(PREPARED),
       '--cnn-checkpoint', str(CNN_DIR/'best.pt'),
       '--catboost-model', str(CAT_DIR/'model.cbm'),
       '--ensemble-config', str(ENSEMBLE_CONFIG),
       '--output-dir', str(EVALUATION_DIAGNOSTICS_DIR),
       '--max-rows', '100000', '--cnn-batch-size', '4096']
subprocess.run(cmd, check=True)
print('Диагностические графики:', sorted(path.name for path in EVALUATION_DIAGNOSTICS_DIR.glob('*.png')))

## 10. Сохранение результатов на Drive

In [ ]:
RESULTS = DRIVE_PROJECT / 'colab_results'
RESULTS.mkdir(parents=True, exist_ok=True)
for source in (CAT_SUBMISSION, CNN_SUBMISSION, ENSEMBLE_SUBMISSION, LOOKUP_SUBMISSION, ENSEMBLE_CONFIG):
    if source.exists(): shutil.copy2(source, RESULTS/source.name)
for source in (CAT_DIR/'metrics.json', CNN_DIR/'history.json', CNN_DIR/'best.pt', CAT_DIR/'model.cbm',
               CAT_CV_DIR/'catboost_cv.json', CNN_CV_DIR/'cnn_cv.json'):
    if source.exists(): shutil.copy2(source, RESULTS/source.name)
for source in (SAVED_EVAL_DIR, CAT_CV_DIR, CNN_CV_DIR, REPORT_ASSETS_DIR, PATTERN_ANALYSIS_DIR, EVALUATION_DIAGNOSTICS_DIR):
    if source.exists(): shutil.copytree(source, RESULTS/source.name, dirs_exist_ok=True)
print('Результаты:', RESULTS)